# Block 1 — Distance between two points

In [ ]:
import numpy as np

def distance(p1,p2):
    return np.linalg.norm(np.array(p1)-np.array(p2))
#np.linalg.norm calculates the straight line distance between two points. This is the only math you need for EAR.

p1=(0,0)
p2=(3,4)

print(distance(p1, p2))

5.0


# Block 2 — EAR function


In [2]:
import numpy as np

# Top part   = eye height (vertical distances)
# Bottom part = eye width  (horizontal distance)

# Eyes open  → height is big  → EAR ≈ 0.35
# Eyes closed → height is small → EAR ≈ 0.0
# Threshold  → EAR < 0.25 → drowsy

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(landmarks,eye_points,w,h):
    # Convert landmark indices to pixel coordinates
    pts=[]
    for idx in eye_points:
        x=int(landmarks[idx].x * w)
        y=int(landmarks[idx].y * h)
        pts.append((x, y))

    # pts[0] = P1, pts[1] = P2 ... pts[5] = P6
    A = distance(pts[1],pts[5])
    B = distance(pts[2], pts[4])
    C = distance(pts[0], pts[3])

    ear=(A+B)/(2.0 * C)#EAR = (|P2-P6| + |P3-P5|) / (2 * |P1-P4|)
    return round(ear, 3)

print("EAR function ready!")

EAR function ready!


# Block 3 — Calculate EAR live from webcam

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
face_mesh=mp_face_mesh.FaceMesh(
    max_num_faces            = 1,
    min_detection_confidence = 0.5,
    min_tracking_confidence  = 0.5
)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(landmarks,eye_points,w,h):
    pts=[(int(landmarks[i].x * w),int(landmarks[i].y*h)) for i in eye_points]
    A=distance(pts[1],pts[5])
    B=distance(pts[2],pts[4])
    C= distance(pts[0],pts[3])
    return round((A+B)/(2.0*C),3)

cap=cv2.VideoCapture(0)

while True:
    ret,frame=cap.read()
    if not ret:
        break

    h,w=frame.shape[:2]
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    results=face_mesh.process(rgb)

    if results.multi_face_landmarks:
        landmarks=results.multi_face_landmarks[0].landmark
        left_ear  = calculate_ear(landmarks, LEFT_EYE,  w, h)
        right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
        avg_ear   = round((left_ear + right_ear) / 2, 3)

        # Show EAR values on screen
        cv2.putText(frame, f"Left EAR:  {left_ear}",  (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Right EAR: {right_ear}", (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Avg EAR:   {avg_ear}",  (20, 110),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("EAR",frame)
    if cv2.waitKey(1) &0xFF==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()




# Block 4 — Add drowsiness detection with counter

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(
    max_num_faces            = 1,
    min_detection_confidence = 0.5,
    min_tracking_confidence  = 0.5
)

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]

EAR_THRESHOLD = 0.25   # below this = eyes closing
FRAME_LIMIT   = 30     # 30 consecutive frames = alert

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(landmarks, eye_points, w, h):
    pts = [(int(landmarks[i].x * w),
            int(landmarks[i].y * h)) for i in eye_points]
    A = distance(pts[1], pts[5])
    B = distance(pts[2], pts[4])
    C = distance(pts[0], pts[3])
    return round((A + B) / (2.0 * C), 3)

# Counter — how many consecutive frames EAR was below threshold
counter = 0
alert   = False

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    h, w = frame.shape[:2]
    rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark

        left_ear  = calculate_ear(landmarks, LEFT_EYE,  w, h)
        right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
        avg_ear   = round((left_ear + right_ear) / 2, 3)

        # Check threshold
        if avg_ear < EAR_THRESHOLD:
            counter+=1
            if counter>=FRAME_LIMIT:
                alert=True
        else:
            counter=0# reset if eyes open
            alert=False
        # Show EAR
        cv2.putText(frame, f"EAR: {avg_ear}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # Show counter
        cv2.putText(frame, f"Counter: {counter}/{FRAME_LIMIT}", (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        # Show alert
        if alert:
            cv2.putText(frame, "DROWSY!", (w//2 - 80, h//2),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
            cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), 5)

        # Color based on status
        color = (0, 0, 255) if avg_ear < EAR_THRESHOLD else (0, 255, 0)
        cv2.putText(frame, f"Status: {'CLOSING' if avg_ear < EAR_THRESHOLD else 'OPEN'}",
                    (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv2.imshow("Drowsiness Detection",frame)
        if cv2.waitKey(1) & 0xFF==ord('q'):
            break

cap.release()
cv2.destroyAllWindows()





<!-- 1. Change FRAME_LIMIT from 30 to 15.
   How does the sensitivity change?
   Does it give more false alerts from blinking?

2. Change EAR_THRESHOLD from 0.25 to 0.20.
   Do you need to close eyes more to trigger it?

3. Print in terminal every time counter resets to 0.
   How many times does it reset per second normally?
   This tells you how often you naturally blink.

4. Add a blink counter — count how many times
   counter goes from >0 back to 0.
   Show it on screen as "Blinks: X" -->

In [6]:
# 1. Change FRAME_LIMIT from 30 to 15.
#    How does the sensitivity change?
#    Does it give more false alerts from blinking?

# 2. Change EAR_THRESHOLD from 0.25 to 0.20.
#    Do you need to close eyes more to trigger it?

# 3. Print in terminal every time counter resets to 0.
#    How many times does it reset per second normally?
#    This tells you how often you naturally blink.

# 4. Add a blink counter — count how many times
#    counter goes from >0 back to 0.
#    Show it on screen as "Blinks: X"